# odmlib v0.2.0: The Value Set System

odmlib v0.2.0 reworks how attribute-level enumerations are validated. Instead
of hard-coding lists inside each model module, every attribute that has a
controlled vocabulary now consults a versioned **value set registry** loaded
from a single bundled JSON file (`odmlib/data/valuesets.json`). This notebook
walks through:

1. The public `ValueSet` API — `value_set`, `validate`, `describe`.
2. The new **regex** entry type (for valid values for attributes like `MetaDataVersion.DefineVersion`).
3. How the registry resolves the right version for each model, including
   cross-version fallback for hybrid extensions.
4. The two patterns a local (custom) model uses to participate in the system —
   subclassing a shipped model to inherit registry entries, and declaring an
   inline value set on the descriptor with `T.ExtendedValidValues`.

Three public class methods on `odmlib.valueset.ValueSet`:

| method                            | returns                                                       |
|-----------------------------------|---------------------------------------------------------------|
| `value_set(attribute, ...)`       | the raw entry — a `list` of strings **or** a regex `dict`     |
| `validate(attribute, value, ...)` | `True` / `False`                                              |
| `describe(attribute, ...)`        | a human-readable description, suitable for error text         |

Each accepts either an explicit `version=` keyword (e.g. `"odm_2_0"`) or an
`instance=` model instance, from which the version is inferred automatically.

## Setup

In [1]:
import odmlib.valueset as VS
import odmlib.odm_1_3_2.model as ODM
import odmlib.odm_2_0.model as ODM2
import odmlib.define_2_1.model as DEFINE
import odmlib.typed as T
from odmlib.exceptions import OdmlibTypeError, OdmlibValidationError

## 1. List-based value sets

Many attributes carry an enumeration of allowed strings. `value_set()` returns
the raw list, `validate()` answers membership, and `describe()` builds a
`"Value must be one of: ..."` message suitable for surfacing in user-facing
errors.

Called with no `version=` or `instance=`, the registry defaults to
`odm_1_3_2` for backward compatibility.

In [2]:
print("value_set :", VS.ValueSet.value_set("StudyEventDef.Repeating"))
print("validate Yes        :", VS.ValueSet.validate("StudyEventDef.Repeating", "Yes"))
print("validate Sometimes  :", VS.ValueSet.validate("StudyEventDef.Repeating", "Sometimes"))
print("describe :", VS.ValueSet.describe("StudyEventDef.Repeating"))

value_set : ['Yes', 'No']
validate Yes        : True
validate Sometimes  : False
describe : Value must be one of: Yes, No


### Same attribute, different model versions

Some attributes diverge between versions. `ItemGroupDef.Repeating` is a
two-value Yes/No flag in ODM 1.3.2 but a four-value enumeration in ODM 2.0.
The registry keeps these versions separate; ask for the one you want.

In [3]:
print("odm_1_3_2 ItemGroupDef.Repeating ->",
      VS.ValueSet.value_set("ItemGroupDef.Repeating", version="odm_1_3_2"))
print("odm_2_0   ItemGroupDef.Repeating ->",
      VS.ValueSet.value_set("ItemGroupDef.Repeating", version="odm_2_0"))

v1_types = VS.ValueSet.value_set("ItemDef.DataType", version="odm_1_3_2")
v2_types = VS.ValueSet.value_set("ItemDef.DataType", version="odm_2_0")
print("\nNew in ODM 2.0 ItemDef.DataType vs 1.3.2 :", sorted(set(v2_types) - set(v1_types)))

odm_1_3_2 ItemGroupDef.Repeating -> ['Yes', 'No']
odm_2_0   ItemGroupDef.Repeating -> ['No', 'Simple', 'Dynamic', 'Static']

New in ODM 2.0 ItemDef.DataType vs 1.3.2 : ['decimal']


## 2. Regex value sets (new in v0.2.0)

Some controlled vocabularies are not practical to enumerate. Define-XML
versions, for instance, ship patch releases whenever CDISC controlled
terminology drops new code lists, so `"2.1.18"` and `"2.1.99"` are equally
valid Define-XML 2.1 version strings. Hard-coded lists aren't a good fit here.

For those attributes the registry stores a dict with a `_regex` pattern and an
optional `_description`:

- `value_set()` returns the dict as-is.
- `validate()` runs `re.fullmatch` (compiled once per `(version, attribute)`
  and cached on the class).
- `describe()` returns the `_description` if present, otherwise a fall-back
  message naming the raw pattern.

In [4]:
entry = VS.ValueSet.value_set("MetaDataVersion.DefineVersion", version="define_2_1")
print("entry type :", type(entry).__name__)
print("entry      :", entry)
print("describe   :", VS.ValueSet.describe("MetaDataVersion.DefineVersion", version="define_2_1"))

entry type : dict
entry      : {'_regex': '^2\\.1(\\.\\d+)?$', '_description': 'Define-XML version 2.1 with optional patch (e.g., 2.1, 2.1.0, 2.1.18)'}
describe   : Define-XML version 2.1 with optional patch (e.g., 2.1, 2.1.0, 2.1.18)


### Trying values against the regex

The Define-XML 2.1 pattern accepts `2.1` and any `2.1.x` patch number, and
rejects everything else: earlier and later major versions, prefix-only
matches, empty strings.

In [5]:
valid   = ["2.1", "2.1.0", "2.1.18", "2.1.99", "2.1.101"]
invalid = ["abc", "1.0", "3.0", "3.0.1", "0.1.0", "2.", "2.1.", ".1", ""]

print("valid values")
for v in valid:
    print(f"  {v!r:>10} -> {VS.ValueSet.validate('MetaDataVersion.DefineVersion', v, version='define_2_1')}")

print("\ninvalid values")
for v in invalid:
    print(f"  {v!r:>10} -> {VS.ValueSet.validate('MetaDataVersion.DefineVersion', v, version='define_2_1')}")

valid values
       '2.1' -> True
     '2.1.0' -> True
    '2.1.18' -> True
    '2.1.99' -> True
   '2.1.101' -> True

invalid values
       'abc' -> False
       '1.0' -> False
       '3.0' -> False
     '3.0.1' -> False
     '0.1.0' -> False
        '2.' -> False
      '2.1.' -> False
        '.1' -> False
          '' -> False


### Regex enforcement on a model instance

The same regex is enforced when the attribute is assigned on a model
instance; the `ValueSetString` descriptor calls `ValueSet.validate()` on every
assignment and raises `OdmlibTypeError` on failure.

In [6]:
mdv = DEFINE.MetaDataVersion(OID="MDV.1", Name="Test Study", DefineVersion="2.1.18")
print("constructed OK with DefineVersion =", mdv.DefineVersion)

try:
    mdv.DefineVersion = "3.0"
except OdmlibTypeError as e:
    print(f"rejected '3.0': {e}")

constructed OK with DefineVersion = 2.1.18
rejected '3.0': Invalid value 3.0 for DefineVersion. Define-XML version 2.1 with optional patch (e.g., 2.1, 2.1.0, 2.1.18)
  Hint: Define-XML version 2.1 with optional patch (e.g., 2.1, 2.1.0, 2.1.18)


### Regex caching

Compiled patterns are stashed on `ValueSet._compiled_regex_cache`, keyed by
`(version, attribute)`, so a hot validation path doesn't re-compile.

In [7]:
VS.ValueSet._compiled_regex_cache.clear()
VS.ValueSet.validate("MetaDataVersion.DefineVersion", "2.1.18", version="define_2_1")
print("cache keys :", list(VS.ValueSet._compiled_regex_cache.keys()))

cache keys : [('define_2_1', 'MetaDataVersion.DefineVersion')]


## 3. Version resolution: how the registry picks the right entry

When you pass `instance=` instead of `version=`, the registry asks
`ValueSetLoader.get_version_for_module()` to translate the instance's class
into a version key. Resolution order:

1. **Exact match** on the module path (e.g. `odmlib.odm_2_0.model` → `odm_2_0`).
2. **Substring marker** for custom modules (any path containing the marker
   `define_2_1` → `define_2_1`).
3. **MRO walk** — for unknown paths, walk the class's base classes and use the
   first one whose module is recognized. This is how local subclasses inherit
   the right version.
4. **Default** to `odm_1_3_2` for backward compatibility.

In [8]:
print("exact module match     ->",
      VS.ValueSetLoader.get_version_for_module("odmlib.odm_2_0.model"))
print("substring marker match ->",
      VS.ValueSetLoader.get_version_for_module("my_sponsor.define_2_1.extensions"))
print("unknown path           ->",
      VS.ValueSetLoader.get_version_for_module("totally.unknown.path"))

exact module match     -> odm_2_0
substring marker match -> define_2_1
unknown path           -> odm_1_3_2


### Automatic version detection from an instance

Passing `instance=` lets the registry pick the version without the caller
spelling it out. Useful when generic code receives a model instance and needs
to look something up about it.

In [9]:
v1_item = ODM.ItemDef(OID="I.1", Name="x", DataType="text", Length=8)
v2_item = ODM2.ItemDef(OID="I.1", Name="x", DataType="decimal", Length=8)

print("v1_item resolves to :", VS.ValueSetLoader.get_version_for_module(
    type(v1_item).__module__, instance_class=type(v1_item)))
print("v2_item resolves to :", VS.ValueSetLoader.get_version_for_module(
    type(v2_item).__module__, instance_class=type(v2_item)))

print("\nv1_item ItemDef.DataType entry length :",
      len(VS.ValueSet.value_set("ItemDef.DataType", instance=v1_item)))
print("v2_item ItemDef.DataType entry length :",
      len(VS.ValueSet.value_set("ItemDef.DataType", instance=v2_item)))

v1_item resolves to : odm_1_3_2
v2_item resolves to : odm_2_0

v1_item ItemDef.DataType entry length : 22
v2_item ItemDef.DataType entry length : 23


### Cross-version fallback

If the resolved version doesn't carry the requested attribute, the registry
searches the other shipped versions in priority order
(`define_2_1, define_2_0, odm_2_0, odm_1_3_2, ct_1_1_1, dataset_1_0_1`) and
returns the first match. This is the mechanism that rescues *hybrid* local
models, an ODM 1.3.2 base extended with Define-XML 2.x attributes such as
`DefineVersion`. Asking for `MetaDataVersion.DefineVersion` under `odm_1_3_2`
still works:

In [10]:
fallback_entry = VS.ValueSet.value_set("MetaDataVersion.DefineVersion", version="odm_1_3_2")
print("falls back to        :", fallback_entry)

try:
    VS.ValueSet.value_set("BogusClass.BogusAttr", version="odm_1_3_2")
except OdmlibValidationError as e:
    print("\ngenuinely unknown attributes still raise:")
    print("  ", e)

falls back to        : {'_regex': '^2\\.1(\\.\\d+)?$', '_description': 'Define-XML version 2.1 with optional patch (e.g., 2.1, 2.1.0, 2.1.18)'}


## 4. Local models and value sets

odmlib's loaders accept a `local_model=True` flag so sponsors can extend the
shipped models with their own additions. Those local classes can plug into the
value set system in two complementary ways:

- **Inherit registry entries** by subclassing a shipped class. The MRO walk
  (and cross-version fallback) gives the local class access to the same
  validation as the parent.
- **Declare a value set inline** on a descriptor with
  `T.ExtendedValidValues(valid_values=[...])`. This is the simplest way to add
  a *new* attribute the registry doesn't know about, or to constrain an
  attribute to a sponsor-specific subset.

### The class name matters

Registry lookups are keyed by `ClassName.AttributeName` (using
`type(instance).__name__`), so a subclass that wants to inherit registry
entries must **keep the same class name** as the parent. A renamed subclass
loses registry resolution for any descriptor inherited from the parent: the
lookup key (e.g. `SponsorStudyEventDef.Repeating`) is registered under no
shipped version, so `ValueSet.value_set()` returns the `UNKNOWN_ATTRIBUTE`
sentinel, `ValueSet.validate()` returns `False`, and the assignment is
rejected with `OdmlibTypeError` — *"No registered value set for ..."*. Note
this is an `OdmlibTypeError`, a *sibling* of `OdmlibValidationError` (both
subclass `OdmlibError`, but neither catches the other), so a handler must
name `OdmlibTypeError` specifically. `ExtendedValidValues` descriptors, by
contrast, carry their valid values on the descriptor itself, so they work
regardless of class name. The two cells below contrast the cases.

In [11]:
# Same name -> registry lookup succeeds.
class StudyEventDef(ODM.StudyEventDef):
    OID = ODM.StudyEventDef.OID
    Name = ODM.StudyEventDef.Name
    Repeating = ODM.StudyEventDef.Repeating
    Type = ODM.StudyEventDef.Type


sed = StudyEventDef(OID="SE.1", Name="Screening", Repeating="No", Type="Scheduled")
print("same-name subclass Repeating accepted:", sed.Repeating)

try:
    StudyEventDef(OID="SE.2", Name="Baseline", Repeating="Sometimes", Type="Scheduled")
except OdmlibTypeError as e:
    print("rejected 'Sometimes':", str(e).splitlines()[0])

same-name subclass Repeating accepted: No
rejected 'Sometimes': Invalid value Sometimes for Repeating. Value must be one of: Yes, No


In [12]:
# Renamed subclass -> registry has no entry under 'SponsorStudyEventDef.Repeating',
# so ValueSet.value_set() returns the UNKNOWN_ATTRIBUTE sentinel, validate()
# returns False, and the assignment is rejected with OdmlibTypeError.
class SponsorStudyEventDef(ODM.StudyEventDef):
    OID = ODM.StudyEventDef.OID
    Name = ODM.StudyEventDef.Name
    Repeating = ODM.StudyEventDef.Repeating
    Type = ODM.StudyEventDef.Type


try:
    SponsorStudyEventDef(OID="SE.1", Name="Screening", Repeating="No", Type="Scheduled")
except OdmlibTypeError as e:
    print("renamed subclass rejected:", str(e).splitlines()[0])

### Pattern A — same-name subclass, inherit registry entries

This is the simplest extension pattern: subclass the shipped class, keep its
name, copy through the descriptors you care about, and rely on the registry
for validation. Validation behaviour is identical to the parent.

### Pattern B — hybrid local model (cross-version fallback)

The `library_define_1_0` extension in this examples repo subclasses ODM 1.3.2
but adds Define-XML 2.x attributes such as `DefineVersion`. The MRO walk
resolves the class to `odm_1_3_2`, then the cross-version fallback locates
`MetaDataVersion.DefineVersion` in the `define_2_1` section and applies its
regex.

In [13]:
class MetaDataVersion(ODM.MetaDataVersion):
    """ODM 1.3.2 base + Define-XML 2.x DefineVersion attribute."""
    OID = ODM.MetaDataVersion.OID
    Name = ODM.MetaDataVersion.Name
    DefineVersion = T.ValueSetString(required=True)


mdv = MetaDataVersion(OID="MDV.1", Name="Hybrid", DefineVersion="2.1.18")
print("hybrid model DefineVersion =", mdv.DefineVersion)

try:
    MetaDataVersion(OID="MDV.2", Name="Hybrid bad", DefineVersion="3.0")
except OdmlibTypeError as e:
    print("rejected '3.0':", str(e).splitlines()[0])

hybrid model DefineVersion = 2.1.18
rejected '3.0': Invalid value 3.0 for DefineVersion. Define-XML version 2.1 with optional patch (e.g., 2.1, 2.1.0, 2.1.18)


### Pattern C — create a value set inside the local model with `ExtendedValidValues`

When the local model adds a *new* attribute, the registry has no entry for it
(and editing the bundled JSON isn't appropriate for sponsor-specific values).
`T.ExtendedValidValues(valid_values=[...])` packages the allowed values right
on the descriptor — no registry interaction, no JSON edits, and the validation
rules ship with the local model itself.

The example below subclasses `StudyEventDef` (same name, so inherited
descriptors still validate via the registry) and adds two sponsor-specific
attributes — `ProtocolPhase` and `Region` — each with its own inline value
set.

In [14]:
# Note: this re-defines StudyEventDef in the notebook namespace, shadowing the
# Pattern A version above. That's intentional. Keeping the same class name is
# how the registry continues to recognize inherited attributes like Repeating.
class StudyEventDef(ODM.StudyEventDef):
    OID = ODM.StudyEventDef.OID
    Name = ODM.StudyEventDef.Name
    Repeating = ODM.StudyEventDef.Repeating          # registry-backed
    Type = ODM.StudyEventDef.Type                    # registry-backed
    ProtocolPhase = T.ExtendedValidValues(           # inline value set
        required=True,
        valid_values=["Screening", "Treatment", "Follow-up"],
    )
    Region = T.ExtendedValidValues(                  # inline value set
        valid_values=["NA", "EU", "APAC"],
    )


ev = StudyEventDef(
    OID="SE.1", Name="Visit 1",
    Repeating="Yes", Type="Scheduled",
    ProtocolPhase="Treatment", Region="EU",
)
print(f"Repeating={ev.Repeating}  Type={ev.Type}  "
      f"ProtocolPhase={ev.ProtocolPhase}  Region={ev.Region}")

Repeating=Yes  Type=Scheduled  ProtocolPhase=Treatment  Region=EU


Both kinds of descriptor enforce their value sets the same way — an invalid
assignment raises `OdmlibTypeError` regardless of whether the rule came from
the registry or the inline list.

In [15]:
# Inline ExtendedValidValues rejects a bad value.
try:
    ev.ProtocolPhase = "Wash-out"
except OdmlibTypeError as e:
    print("inline rejected     :", str(e).splitlines()[0])

# Registry-backed ValueSetString rejects a bad value.
try:
    ev.Repeating = "Sometimes"
except OdmlibTypeError as e:
    print("registry rejected   :", str(e).splitlines()[0])

inline rejected     : Invalid value Wash-out for ProtocolPhase. Value must be one of Screening, Treatment, Follow-up
registry rejected   : Invalid value Sometimes for Repeating. Value must be one of: Yes, No


### Picking between the patterns

| You want to ...                                                           | Use                                              |
|---------------------------------------------------------------------------|--------------------------------------------------|
| Reuse a shipped attribute's value set unchanged                           | Subclass with same name, copy descriptor through |
| Add a Define-XML attribute to an ODM 1.3.2 local model                    | `T.ValueSetString(required=True)` (Pattern B)    |
| Add a sponsor-specific attribute the registry doesn't know about          | `T.ExtendedValidValues(valid_values=[...])`      |
| Narrow an existing attribute to a sponsor-specific subset of values       | Override with `T.ExtendedValidValues(...)`       |
| Add or change a value set across many sponsors / standard releases        | Edit `odmlib/data/valuesets.json` (rare)         |

## 5. Summary

- `ValueSet.value_set / validate / describe` are the three public hooks; pass
  either `version=` or `instance=` to target the right model.
- **List entries** are unchanged from earlier versions of odmlib. **Regex
  entries** (dicts with `_regex` and optional `_description`) are new in v0.2.0
  and allow patterns that track moving CDISC standards such as the Define-XML
  version string.
- Version resolution walks: exact module match → substring marker → MRO of
  the instance class → cross-version fallback over the other shipped versions.
- Local models join the system in two ways:
  - Subclass a shipped class **using the same class name** to inherit
    registry entries (Patterns A and B).
  - Declare `T.ExtendedValidValues(valid_values=[...])` on the descriptor to
    carry the value set inline (Pattern C). This is the right tool for new
    sponsor-specific attributes.